# 28C0R · Full 120-condition launch and matched-pool learned-feature ablation

Scope: six datasets (ETTh1, Weather, Electricity, Traffic, Exchange, Solar), horizons {96,192,336,720}, five forecasters (DLinear, PatchTST, iTransformer, TimeMixer, Seg-MoE). Step 1 builds a **120-row inventory**, including checkpoint and pair-cache candidates; never retrains a forecaster, never silently fills a missing condition. Step 2 executes one reproducible, same-candidate-pool relevance ablation on Electricity H192 PatchTST OOF records already created by 28B1. Two architecture-matched rankers see the same 96 candidates, labels and training protocol. Their sole difference is one learned-encoder similarity feature (column 3): baseline observable columns 0–2 and 4–31 versus augmented columns 0–31. Column 0 is the original seven-statistics reranker score; columns 4–31 contain four blocks of seven scaled statistics. Forecaster geometry columns 32–43 are excluded from both models.

Chronological training on F1, inner epoch selection on F2, untouched F3 evaluation. Report reference reranker and both learned rankers by Top-10 *AnalogFutureMSE* from OOF pair futures. This one-condition pilot tests whether the additional learned feature helps candidate relevance; it does not recreate the PRR-Stat checkpoint. A separate Top-M audit estimates opportunities missed by Pattern Top-100; it is **not** a five-forecaster outcome, downstream forecast MSE, or an independent test estimate. The inventory is a launch checklist for filling all 120 rows with matched direct/reference/augmented forecast results in later runs.


In [ ]:
from pathlib import Path
import re,json,time,zipfile,ast,struct,gc,os
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
ROOT=Path(os.environ.get('PRR_ROOT', str(Path.cwd().resolve().parents[1] if Path.cwd().name == 'prr_long_horizon' else Path.cwd().resolve())))
STRONG=Path('/data/dataset/strong_forecaster')
LEGACY_STRONG_ROOT=Path('/data/code/stock_regime_retrieval/strong_forecaster')
OUT=ROOT/'results/28C0_full120_learned_feature_launch';OUT.mkdir(parents=True,exist_ok=True)
DATASETS=('ETTh1','Weather','Electricity','Traffic','Exchange','Solar')
HORIZONS=(96,192,336,720)
BACKBONES=('DLinear','PatchTST','iTransformer','TimeMixer','Seg-MoE')
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
START=time.perf_counter();MAX_SECONDS=1500
print('Device',DEVICE,'Result folder',OUT)


## 1. Inventory all 120 conditions

This scan collects filenames and headers under established strong-forecaster roots. A filename match is only a discovery hint; row status stays `REVIEW_REQUIRED` until exact checkpoint identity, pair alignment, candidate pool, and forecast normalization are checked. A cache marked test by name is **listed but not opened or evaluated**. No `torch.load` or full `.npz` array read occurs in this inventory.


In [ ]:
import itertools
scan_roots=[p for p in (STRONG,ROOT/'results',ROOT/'checkpoints',LEGACY_STRONG_ROOT) if p.is_dir()]
ALLOWED={'.npz','.pt','.pth','.csv'}
paths=[]
for base in scan_roots:
 for p in base.rglob('*'):
  if p.is_file() and p.suffix.lower() in ALLOWED:
   # avoid traversal of giant, unrelated datasets through repository root
   paths.append(p)
print('Relevant root count',len(scan_roots),'artifact paths',len(paths))
def detect(path,options):
 s=path.name.lower();v=[]
 for x in options:
  alias=[x.lower()]
  if x=='Seg-MoE':alias=['seg-moe','segmoe','seg_moe']
  if x=='DLinear':alias=['dlinear','d_linear']
  if x=='iTransformer':alias=['itransformer','i_transformer']
  if any(re.search(r'(?<![a-z])'+re.escape(q)+r'(?![a-z])',s) for q in alias):v.append(x)
 return v[0] if len(v)==1 else None
def horizon(path):
 s=path.name.lower()
 hit=[h for h in HORIZONS if re.search(r'(?<![a-z0-9])h'+str(h)+r'(?![0-9])|(?<![0-9])_'+str(h)+r'_(?![0-9])',s)]
 return hit[0] if len(hit)==1 else None
by_condition={key:[] for key in itertools.product(DATASETS,HORIZONS,BACKBONES)}
for p in paths:
 ds=detect(p,DATASETS);backbone=detect(p,BACKBONES);h=horizon(p)
 if (ds,h,backbone) in by_condition:by_condition[(ds,h,backbone)].append(str(p))
manifest=[]
for (ds,h,bb),hits in by_condition.items():
 ck=[p for p in hits if Path(p).suffix.lower() in ('.pt','.pth')]
 pair=[p for p in hits if Path(p).suffix.lower()=='.npz' and any(word in Path(p).name.lower() for word in ('pair','direct'))]
 validation=[p for p in pair if 'validation' in Path(p).name.lower()]
 manifest.append(dict(Dataset=ds,Horizon=h,Forecaster=bb,CheckpointHints=len(ck),
                      PairCacheHints=len(pair),ValidationCacheHints=len(validation),
                      Status='REVIEW_REQUIRED',CheckpointPaths=json.dumps(ck),
                      ValidationCachePaths=json.dumps(validation),AllHintPaths=json.dumps(hits)))
manifest=pd.DataFrame(manifest)
assert len(manifest)==120 and len(manifest[['Dataset','Horizon','Forecaster']].drop_duplicates())==120
display(manifest.groupby('Forecaster')[['CheckpointHints','PairCacheHints','ValidationCacheHints']].apply(lambda d:(d>0).sum()))
display(manifest[['Dataset','Horizon','Forecaster','CheckpointHints','ValidationCacheHints','Status']].head(30))
manifest.to_csv(OUT/'full120_artifact_inventory.csv',index=False)
print('Exactly 120 conditions inventoried; file hints are not verified results.')


## 2. Load the frozen, aligned OOF candidate sets

Each 28B1 `.npz` supplies selected candidate anchors, 44 inference-observable columns, and reference reranker score. The matching 28B0 fold direct cache supplies the *same* query anchor/channel order and normalized direct/true residual. Candidate futures are reconstructed from the original Electricity CSV, using exactly the fold-prefix normalization and historical candidate anchors. No test or validation sample is accessed here.


In [ ]:
B1=ROOT/'results/28B1_electricity_patchtst_h192_candidate_utility_cache'
B0=ROOT/'results/28B0_electricity_patchtst_h192_fold_direct_cache'
DATA_CANDIDATES=[Path('/data/Time-Series-Library/dataset/electricity/electricity.csv'),Path('/data/Time-Series-Library_v2/dataset/electricity/electricity.csv'),Path('/code/Time-Series-Library/dataset/electricity/electricity.csv'),Path('/data/dataset/electricity/electricity.csv')]
data_path=next((p for p in DATA_CANDIDATES if p.is_file()),None)
if data_path is None:raise FileNotFoundError('Electricity source CSV missing')
df=pd.read_csv(data_path)
for col in list(df):
 if str(col).lower() in ('date','datetime','timestamp','time'):df=df.drop(columns=[col])
if 'OT' in df.columns:df=df[[k for k in df if k!='OT']+['OT']]
df=df.apply(pd.to_numeric,errors='coerce').dropna(axis=1,how='all').replace([np.inf,-np.inf],np.nan).interpolate(axis=0,limit_direction='both').ffill().bfill()
raw=df.to_numpy(np.float32)
if raw.shape!=(26304,321):raise ValueError(f'Unexpected Electricity shape: {raw.shape}')
def fold_paths(f):return (B1/f'Electricity_PatchTST_H192_F{f}_utility.npz',B0/f'PatchTST_Electricity_H192_F{f}_pair_direct.npz')
def load_fold(f):
 cp,dp=fold_paths(f)
 for p in (cp,dp):
  if not p.is_file():raise FileNotFoundError(f'Need previous 28B0/28B1 OOF cache: {p}')
 with np.load(cp,allow_pickle=False) as obj:
  required={'feature','base','selected_anchor','anchor','channel','prefix','fold'}
  if not required.issubset(obj.files):raise ValueError(f'Missing candidate keys: {cp}')
  x={k:np.asarray(obj[k]) for k in required}
 with np.load(dp,allow_pickle=False) as obj:
  a=obj['anchor'];ch=obj['channel'];direct=obj['direct_residual'];true=obj['true_residual']
  if int(obj['fold'])!=f or int(obj['prefix'])!=int(x['prefix']) or int(obj['horizon'])!=192:raise ValueError('OOF direct cache identity mismatch')
 if int(x['fold'])!=f or x['feature'].shape!=(2048,96,44) or x['base'].shape!=(2048,96) or x['selected_anchor'].shape!=(2048,96):raise ValueError('28B1 candidate schema mismatch')
 if not np.array_equal(x['anchor'],a) or not np.array_equal(x['channel'],ch):raise ValueError('OOF pair order mismatch')
 if direct.shape!=true.shape!=(2048,192):raise ValueError('OOF horizon shape mismatch')
 if not np.isfinite(x['feature']).all() or not np.isfinite(true).all():raise ValueError('Nonfinite inputs')
 prefix=int(x['prefix']);mean=raw[:prefix].mean(axis=0);std=raw[:prefix].std(axis=0)
 if np.any(std<=1e-6):raise ValueError('Degenerate fold normalization')
 z=((raw-mean)/std).astype(np.float32)
 if np.any(x['selected_anchor']+192>prefix):raise ValueError('Candidate future crosses fold boundary')
 check_true=z[a[:,None]+np.arange(192)[None,:],ch[:,None]]-z[a-1,ch][:,None]
 if not np.allclose(check_true,true,rtol=6e-4,atol=6e-4):raise ValueError('Direct true future normalization mismatch')
 # 32-query blocks keep candidate trajectory memory bounded.
 label=np.empty((2048,96),np.float32)
 for left in range(0,2048,32):
  sel=x['selected_anchor'][left:left+32]
  cc=ch[left:left+32]
  fut=z[sel[:,:,None]+np.arange(192)[None,None,:],cc[:,None,None]]
  cand=fut-z[sel-1,cc[:,None]][:,:,None]
  label[left:left+32]=((cand-true[left:left+32,None,:])**2).mean(axis=-1)
 if not np.isfinite(label).all():raise ValueError('Invalid analog future labels')
 return {'feature':x['feature'].astype(np.float32),'base':x['base'].astype(np.float32),'loss':label,
         'anchor':a.astype(np.int64),'prefix':prefix}
folds={f:load_fold(f) for f in (1,2,3)}
display(pd.DataFrame([{'fold':f,'pairs':len(x['anchor']),'prefix':x['prefix'],
                       'old_top10_analog_mse':float(np.take_along_axis(x['loss'],np.argsort(-x['base'],axis=1)[:,:10],axis=1).mean())}
                      for f,x in folds.items()]))


## 3. Top-M candidate opportunity audit

On 256 chronological F3 OOF pairs, compare Pattern Top-100 and learned-embedding Top-100 using the same historical memory. The comparison reports learned-only candidates and oracle Top-10 AnalogFutureMSE within Pattern100 versus their union. Future labels are used only to evaluate availability, never to select inference candidates. Oracle gains are ceilings, not realized reranker or forecaster improvements. F3 is diagnostic only.


In [ ]:
import math,random,gc
import torch.nn.functional as F
EPS=1e-8
class ProbeCfg:
 retrieval_seq_len=96;memory_stride=8;top_k=10;top_m_pattern=100;top_m_learned=100
 emb_dim=64;enc_hidden=128;rerank_hidden=384;rerank_dropout=.1;feature_clip=8.;seed=0
cfg=ProbeCfg()
MAX_TOTAL_SECONDS=1500
def eval_anchors(start, end, horizon, stride=1, lookback=None):
    if lookback is None:
        lookback = cfg.retrieval_seq_len

    return np.arange(
        max(int(start), int(lookback)),
        int(end) - int(horizon) + 1,
        int(stride),
        dtype=np.int64,
    )

def batch_pattern(x):
    x = np.asarray(x, dtype=np.float32)
    xc = x - x.mean(axis=-1, keepdims=True)
    n = np.linalg.norm(xc, axis=-1, keepdims=True)

    return np.where(
        n > EPS,
        xc / np.maximum(n, EPS),
        0.0,
    ).astype(np.float32)

def context7(x):
    x = np.asarray(x, dtype=np.float32)
    short = max(8, cfg.retrieval_seq_len // 4)

    m = x.mean(axis=-1)
    s = x.std(axis=-1) + EPS

    f1 = (x[..., -1] - m) / s
    f2 = (x[..., -short:].mean(axis=-1) - m) / s
    f3 = (x[..., -1] - x[..., -short]) / s
    f4 = (x[..., -1] - x[..., 0]) / s

    df = np.diff(x, axis=-1)
    ds = np.diff(x[..., -short:], axis=-1)

    f5 = (
        ds.std(axis=-1) + EPS
    ) / (
        df.std(axis=-1) + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        cfg.retrieval_seq_len,
        dtype=np.float32,
    )

    t = t - t.mean()

    f6 = (
        np.sum(
            t * (x - m[..., None]),
            axis=-1,
        )
        / (
            np.sum(t * t) + EPS
        )
    ) / s

    a = x[..., :-1]
    b = x[..., 1:]

    a = a - a.mean(axis=-1, keepdims=True)
    b = b - b.mean(axis=-1, keepdims=True)

    f7 = (
        np.sum(a * b, axis=-1)
        / (
            np.sqrt(
                np.sum(a * a, axis=-1)
                * np.sum(b * b, axis=-1)
            )
            + EPS
        )
    )

    return np.stack(
        [f1, f2, f3, f4, f5, f6, f7],
        axis=-1,
    ).astype(np.float32)

def extract_channel(
    z,
    channel,
    anchors,
    horizon,
):
    anchors = np.asarray(anchors, dtype=np.int64)

    pi = (
        anchors[:, None]
        - cfg.retrieval_seq_len
        + np.arange(cfg.retrieval_seq_len)[None, :]
    )

    fi = (
        anchors[:, None]
        + np.arange(int(horizon))[None, :]
    )

    past = z[pi, channel].astype(np.float32)
    future = z[fi, channel].astype(np.float32)
    current = z[anchors - 1, channel].astype(np.float32)

    return (
        past,
        (future - current[:, None]).astype(np.float32),
    )

class TemporalEncoder1D(nn.Module):
    def __init__(self, hidden=128, emb_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(
                1,
                hidden // 2,
                5,
                stride=2,
                padding=2,
            ),
            nn.GELU(),
            nn.Conv1d(
                hidden // 2,
                hidden,
                5,
                stride=2,
                padding=2,
            ),
            nn.GELU(),
            nn.Conv1d(
                hidden,
                hidden,
                3,
                stride=2,
                padding=1,
            ),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.proj = nn.Linear(
            hidden,
            emb_dim,
        )

    def forward(self, x):
        if x.ndim == 2:
            x = x[:, None, :]

        h = self.net(x).squeeze(-1)

        return F.normalize(
            self.proj(h),
            dim=-1,
        )

def inverse_softplus(x):
    return math.log(
        math.exp(float(x)) - 1.0
    )

class WideFusionReranker(nn.Module):
    def __init__(self):
        super().__init__()

        in_dim = 1 + 4 * (
            7 + cfg.emb_dim
        )

        h = cfg.rerank_hidden

        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h, 1),
        )

        self.raw_alpha = nn.Parameter(
            torch.tensor(
                inverse_softplus(0.1),
                dtype=torch.float32,
            )
        )

    def forward(self, feat, base, mask):
        delta = self.net(feat).squeeze(-1)
        alpha = F.softplus(self.raw_alpha)

        score = (
            base
            + alpha * delta
        )

        return score.masked_fill(
            ~mask,
            -1e9,
        )

def make_encoder_pair(seed):
    seed_everything(seed)

    qenc = TemporalEncoder1D(
        cfg.enc_hidden,
        cfg.emb_dim,
    ).to(DEVICE)

    henc = TemporalEncoder1D(
        cfg.enc_hidden,
        cfg.emb_dim,
    ).to(DEVICE)

    return qenc, henc

def make_reranker(seed):
    seed_everything(seed)

    return WideFusionReranker().to(DEVICE)

@torch.no_grad()
def encode_rows(enc, x, batch=512):
    out = []
    enc.eval()

    for left in range(0, len(x), batch):
        t = torch.from_numpy(
            x[left:left+batch]
        ).float().to(DEVICE)

        out.append(
            enc(t).cpu().numpy()
        )

    return np.concatenate(
        out,
        axis=0,
    ).astype(np.float32)
def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_torch(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )

class UpgradedStack:
    def __init__(
        self,
        qenc,
        henc,
        reranker,
        stats_med,
        stats_iqr,
        dataset,
        horizon,
        training_channels,
        training_query_anchors,
        rerank_best_epoch,
    ):
        self.qenc = qenc.eval()
        self.henc = henc.eval()
        self.reranker = reranker.eval()

        self.stats_med = np.asarray(
            stats_med,
            dtype=np.float32,
        )

        self.stats_iqr = np.asarray(
            stats_iqr,
            dtype=np.float32,
        )

        self.dataset = str(dataset)
        self.horizon = int(horizon)

        self.training_channels = np.asarray(
            training_channels,
            dtype=np.int64,
        )

        self.training_query_anchors = np.asarray(
            training_query_anchors,
            dtype=np.int64,
        )

        self.rerank_best_epoch = int(
            rerank_best_epoch
        )

    def scale_stats(self, x):
        return np.clip(
            (
                x
                - self.stats_med
            )
            / self.stats_iqr,
            -cfg.feature_clip,
            cfg.feature_clip,
        ).astype(np.float32)

def load_stack(path):
    obj = load_torch(path)

    qenc, henc = make_encoder_pair(
        cfg.seed
    )

    reranker = make_reranker(
        cfg.seed
    )

    qenc.load_state_dict(
        obj["query_encoder"]
    )

    henc.load_state_dict(
        obj["history_encoder"]
    )

    reranker.load_state_dict(
        obj["reranker"]
    )

    return UpgradedStack(
        qenc=qenc,
        henc=henc,
        reranker=reranker,
        stats_med=obj["stats_med"],
        stats_iqr=obj["stats_iqr"],
        dataset=obj["dataset"],
        horizon=obj["horizon"],
        training_channels=obj["training_channels"],
        training_query_anchors=obj["training_query_anchors"],
        rerank_best_epoch=obj["rerank_best_epoch"],
    )

def prefix_normalize(
    raw,
    prefix,
):
    prefix = int(prefix)

    mean = raw[
        :prefix
    ].mean(
        axis=0
    ).astype(
        np.float32
    )

    std = raw[
        :prefix
    ].std(
        axis=0,
        ddof=0,
    ).astype(
        np.float32
    )

    bad = np.flatnonzero(
        std <= 1e-6
    )

    if len(bad):
        raise ValueError(
            f"Prefix-degenerate channels: "
            f"{bad[:40].tolist()}"
        )

    z = (
        (
            raw
            - mean[
                None,
                :,
            ]
        )
        / std[
            None,
            :,
        ]
    ).astype(
        np.float32
    )

    return (
        z,
        mean,
        std,
    )

def topk_np(scores, k):
    k = min(
        int(k),
        len(scores),
    )

    part = np.argpartition(
        -scores,
        k - 1,
    )[:k]

    return part[
        np.argsort(
            -scores[part]
        )
    ]

@torch.no_grad()
def build_memory_features(
    data,
    zarr,
    horizon,
    stack,
    boundary,
):
    horizon = int(
        horizon
    )

    boundary = int(
        boundary
    )

    C = data[
        "n_channels"
    ]

    memory_anchors = np.arange(
        cfg.retrieval_seq_len,
        boundary
        - horizon
        + 1,
        cfg.memory_stride,
        dtype=np.int64,
    )

    if (
        len(
            memory_anchors
        )
        < cfg.top_k
    ):
        raise ValueError(
            "Insufficient admissible historical memory."
        )

    M = len(
        memory_anchors
    )

    pattern = np.empty(
        (
            C,
            M,
            cfg.retrieval_seq_len,
        ),
        dtype=np.float32,
    )

    stats = np.empty(
        (
            C,
            M,
            7,
        ),
        dtype=np.float32,
    )

    emb = np.empty(
        (
            C,
            M,
            cfg.emb_dim,
        ),
        dtype=np.float32,
    )

    for c in range(
        C
    ):
        if time.perf_counter()-START>MAX_TOTAL_SECONDS: raise RuntimeError("Global time guard reached during memory build")
        past, _ = extract_channel(
            zarr,
            c,
            memory_anchors,
            horizon,
        )

        pattern[
            c
        ] = batch_pattern(
            past
        )

        stats[
            c
        ] = stack.scale_stats(
            context7(
                past
            )
        )

        emb[
            c
        ] = encode_rows(
            stack.henc,
            past,
        )

        if (
            c == 0
            or (
                c
                + 1
            )
            % 100
            == 0
            or c
            + 1
            == C
        ):
            print(
                f"{data['name']} boundary={boundary}: "
                f"memory {c+1}/{C}"
            )

    return {
        "anchors":
            memory_anchors,
        "pattern":
            pattern,
        "stats":
            stats,
        "emb":
            emb,
        "M":
            M,
        "boundary":
            boundary,
    }

In [ ]:
STACK=ROOT/'checkpoints/14_matched_original_trust_h192/fold_stacks/Electricity_H192_F3_screen_budget.pt'
if not STACK.is_file():raise FileNotFoundError(STACK)
stack=load_stack(STACK)
for m in (stack.qenc,stack.henc,stack.reranker):m.to(DEVICE).eval()
fold3=folds[3];prefix=fold3['prefix']
mu=raw[:prefix].mean(axis=0);sd=raw[:prefix].std(axis=0)
z=((raw-mu)/sd).astype(np.float32)
data={'name':'Electricity','n_channels':321}
memory=build_memory_features(data,z,192,stack,prefix)
SAMPLE=256
select=np.linspace(0,len(fold3['anchor'])-1,SAMPLE,dtype=np.int64)
query_anchor=fold3['anchor'][select]
with np.load(fold_paths(3)[1],allow_pickle=False) as zz:
 query_channel=np.asarray(zz['channel'])[select]
 query_true=np.asarray(zz['true_residual'])[select]
results=[]
for left in range(0,SAMPLE,16):
 if time.perf_counter()-START>MAX_SECONDS:raise TimeoutError('Top-M audit time guard')
 aa=query_anchor[left:left+16];cc=query_channel[left:left+16];yy=query_true[left:left+16]
 pi=aa[:,None]-96+np.arange(96)[None,:]
 past=z[pi,cc[:,None]].astype(np.float32)
 qp=batch_pattern(past);qe=encode_rows(stack.qenc,past)
 pat=np.einsum('bl,bml->bm',qp,memory['pattern'][cc])
 learned=np.einsum('be,bme->bm',qe,memory['emb'][cc])
 for j in range(len(aa)):
  pidx=topk_np(pat[j],100);lidx=topk_np(learned[j],100)
  union=np.union1d(pidx,lidx);new_only=np.setdiff1d(lidx,pidx)
  hist_anchor=memory['anchors'][union]
  if np.any(hist_anchor+192>prefix):raise ValueError('Candidate future leakage')
  future=z[hist_anchor[:,None]+np.arange(192)[None,:],cc[j]]-z[hist_anchor-1,cc[j]][:,None]
  loss=((future-yy[j][None,:])**2).mean(axis=1)
  is_p=np.isin(union,pidx)
  pat_loss=loss[is_p]
  results.append(dict(anchor=int(aa[j]),channel=int(cc[j]),union_size=len(union),new_only_count=len(new_only),
    best_is_outside_pattern=bool(not is_p[int(np.argmin(loss))]),
    pattern_oracle_top10=float(np.partition(pat_loss,9)[:10].mean()),
    union_oracle_top10=float(np.partition(loss,9)[:10].mean())))
coverage=pd.DataFrame(results)
coverage['improvement_available']=coverage.pattern_oracle_top10-coverage.union_oracle_top10
coverage.to_csv(OUT/'f3_topm_opportunity_256pairs.csv',index=False)
display(coverage.describe())
print('Best candidate outside Pattern100:',coverage.best_is_outside_pattern.mean())
print('Oracle Top10 mean: Pattern100',coverage.pattern_oracle_top10.mean(),'union',coverage.union_oracle_top10.mean())
print('Oracle improvement is only an availability upper bound')
del stack,memory,z;gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


## 3. Matched architecture: 31 columns vs 32 columns

Baseline uses candidate rank, pattern similarity and four 7-stat blocks. The reference Exp14 score is excluded because it uses learned embeddings; this baseline is an architecture-matched feature ablation, not the exact original-paper checkpoint. Augmented additionally uses column 3 (learned dual-encoder similarity). Both see exactly the same labels and pool. Feature normalization fits F1 only. The listwise target is normalized per query to avoid differences in fold scale. The F2 metric chooses an epoch separately for each variant; F3 is not used for model selection.


In [ ]:
SEED=28030;BATCH=32;MAX_EPOCHS=16;PATIENCE=4;TOPK=10
COLS={'7stat_and_pattern':tuple([1,2]+list(range(4,32))),
      'plus_learned_similarity':tuple(range(1,32))}
assert len(COLS['7stat_and_pattern'])==30 and len(COLS['plus_learned_similarity'])==31
class Ranker(nn.Module):
 def __init__(self,d):
  super().__init__();self.net=nn.Sequential(nn.Linear(d,128),nn.LayerNorm(128),nn.GELU(),nn.Dropout(.1),
    nn.Linear(128,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1))
 def forward(self,x):return self.net(x).squeeze(-1)
def normalize(loss):
 mu=loss.mean(dim=1,keepdim=True);sd=loss.std(dim=1,unbiased=False,keepdim=True).clamp_min(1e-6)
 return F.softmax(-((loss-mu)/sd)/.5,dim=1)
def rank_loss(score,y):return -(normalize(y)*F.log_softmax(score,dim=1)).sum(dim=1).mean()
def fit_scaler(f,cols):
 a=f[:,:,list(cols)].reshape(-1,len(cols));med=np.median(a,axis=0).astype(np.float32)
 spread=(np.percentile(a,75,axis=0)-np.percentile(a,25,axis=0)).astype(np.float32)
 return med,np.where(spread<1e-5,1.,spread).astype(np.float32)
def scaled(f,cols,med,spread):return np.clip((f[:,:,list(cols)]-med[None,None,:])/spread[None,None,:],-8,8).astype(np.float32)
def batches(x,y,indices):
 for left in range(0,len(indices),BATCH):
  ids=indices[left:left+BATCH]
  yield torch.as_tensor(x[ids],device=DEVICE,dtype=torch.float32),torch.as_tensor(y[ids],device=DEVICE,dtype=torch.float32)
@torch.no_grad()
def evaluate(model,x,y):
 model.eval();scores=[]
 for bx,by in batches(x,y,np.arange(len(x))):scores.append(model(bx).cpu().numpy())
 pred=np.concatenate(scores)
 ids=np.argsort(-pred,axis=1)[:,:TOPK]
 return float(np.take_along_axis(y,ids,axis=1).mean()),pred

def train_variant(name,cols):
 med,spread=fit_scaler(folds[1]['feature'],cols)
 x1=scaled(folds[1]['feature'],cols,med,spread);y1=folds[1]['loss']
 x2=scaled(folds[2]['feature'],cols,med,spread);y2=folds[2]['loss']
 torch.manual_seed(SEED)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
 model=Ranker(len(cols)).to(DEVICE)
 optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
 rng=np.random.default_rng(SEED)
 best=float('inf');best_state=None;best_epoch=None;stale=0;history=[]
 for ep in range(1,MAX_EPOCHS+1):
  if time.perf_counter()-START>MAX_SECONDS:raise TimeoutError('28C0 time guard reached')
  model.train();order=rng.permutation(len(x1))
  for bx,by in batches(x1,y1,order):
   err=rank_loss(model(bx),by)
   optimizer.zero_grad(set_to_none=True);err.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()
  val, _=evaluate(model,x2,y2);history.append(dict(epoch=ep,F2_top10_analog_mse=val))
  print(name,'epoch',ep,'F2',round(val,6),flush=True)
  if val<best-1e-6:
   best=val;best_epoch=ep;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};stale=0
  else:
   stale+=1
   if stale>=PATIENCE:break
 model.load_state_dict(best_state)
 x3=scaled(folds[3]['feature'],cols,med,spread);f3,pred=evaluate(model,x3,folds[3]['loss'])
 obj={'name':name,'columns':list(cols),'state_dict':best_state,'feature_median':med,'feature_iqr':spread,
      'best_epoch_F2':best_epoch,'F2_mse':best,'F3_mse':f3,'history':history,
      'training_fold':1,'epoch_selection_fold':2,'evaluation_fold':3}
 tmp=OUT/f'{name}.partial.pt';torch.save(obj,tmp);os.replace(tmp,OUT/f'{name}.pt')
 pd.DataFrame(history).to_csv(OUT/f'{name}_training.csv',index=False)
 return f3,best_epoch


In [ ]:
result=[]
for name,cols in COLS.items():
 mse,epoch=train_variant(name,cols)
 result.append(dict(policy=name,F3_top10_analog_future_mse=mse,best_epoch_F2=epoch))
x=folds[3];ids=np.argsort(-x['base'],axis=1)[:,:TOPK]
result.insert(0,dict(policy='Exp14_learned_stack_score_reference',F3_top10_analog_future_mse=float(np.take_along_axis(x['loss'],ids,axis=1).mean()),best_epoch_F2=np.nan))
result.append(dict(policy='candidate_pool_oracle_ceiling',F3_top10_analog_future_mse=float(np.partition(x['loss'],TOPK-1,axis=1)[:,:TOPK].mean()),best_epoch_F2=np.nan))
report=pd.DataFrame(result);display(report)
report.to_csv(OUT/'electricity_h192_patchtst_matched_pool_ablation.csv',index=False)
base=float(report.loc[report.policy=='7stat_and_pattern','F3_top10_analog_future_mse'].iloc[0]);add=float(report.loc[report.policy=='plus_learned_similarity','F3_top10_analog_future_mse'].iloc[0])
print('F3 learned feature minus matched baseline:',round(add-base,7),'(negative is improvement)')
print('Full condition manifest:',OUT/'full120_artifact_inventory.csv')
print('Elapsed seconds:',round(time.perf_counter()-START,1))
print('NEXT: verify checkpoint/pair identity for all 120 manifest rows; compare old/new policies under matching candidate pool, direct caches and validation-chosen integration. No test used here.')
